# 03 · Train (medium run)

Kaggle: Settings → Accelerator → **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All (Commit)** so it runs in the background. When it finishes, open the version's log and paste the last cell's output back to Claude.

In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel
import os, time, json, random, gc
from bisect import bisect_right
import torch, transformers, unsloth
assert torch.cuda.is_available(), "No GPU. Kaggle: Settings > Accelerator > GPU T4 x2 (and Internet: On). Colab: Runtime > Change runtime type > T4 GPU"

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
report = {
    "gpu": torch.cuda.get_device_name(0),
    "versions": {"unsloth": unsloth.__version__, "transformers": transformers.__version__, "torch": torch.__version__},
}
print(report)

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files

DATA_REPO = "theatticusproject/cuad-qa"
DATA_REV = "refs/convert/parquet"
paths = [
    hf_hub_download(DATA_REPO, f, repo_type="dataset", revision=DATA_REV)
    for f in list_repo_files(DATA_REPO, repo_type="dataset", revision=DATA_REV)
    if f.startswith("default/train/") and f.endswith(".parquet")
]
train = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
train["answer_texts"] = train["answers"].apply(lambda a: list(a["text"]))
train["answer_starts"] = train["answers"].apply(lambda a: [int(s) for s in a["answer_start"]])
train["has_answer"] = train["answer_texts"].apply(len) > 0
train = train.drop(columns=["answers"])
print(len(train), "rows")

In [ ]:
BASE = "unsloth/llama-3-8b-bnb-4bit"
MAX_TOKENS = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE,
    max_seq_length=MAX_TOKENS,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
WINDOW = 512
MAX_WINDOW = 1024
EXTRA_NEG_FROM_UNANSWERED = 1500
EXTRA_NEG_FROM_ANSWERED = 1500
VAL_CONTRACTS = 20
VAL_EXAMPLES = 200
NOT_FOUND = "Not found."
ALPACA = (
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:\n"
)
INSTRUCTION = (
    "{question}\n"
    "Quote the exact text from the contract excerpt that answers this, one passage per line. "
    'If there is none, answer "Not found."'
)

contracts = train.drop_duplicates("title").set_index("title")["context"].to_dict()
token_starts, token_ends = {}, {}
for t, c in contracts.items():
    offsets = tokenizer(c, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    token_starts[t] = [s for s, _ in offsets]
    token_ends[t] = [e for _, e in offsets]

titles = sorted(contracts)
random.Random(SEED).shuffle(titles)
val_titles = set(titles[:VAL_CONTRACTS])

def token_index(t, char):
    return max(0, bisect_right(token_starts[t], char) - 1)

def answer_spans(row):
    return [(s, s + len(a), a) for a, s in zip(row.answer_texts, row.answer_starts)]

def make(row, start, end):
    a, b = token_starts[row.title][start], token_ends[row.title][end - 1]
    inside = []
    for s, e, text in answer_spans(row):
        if a <= s and e <= b:
            inside.append((s, text))
        elif s < b and a < e:
            return None
    answers = list(dict.fromkeys(text for _, text in sorted(inside)))
    return {
        "title": row.title,
        "instruction": INSTRUCTION.format(question=row.question),
        "input": contracts[row.title][a:b],
        "output": "\n".join(answers) if answers else NOT_FOUND,
    }

def positive(row, rng):
    t = row.title
    n = len(token_starts[t])
    s, e, _ = answer_spans(row)[0]
    i0, i1 = token_index(t, s), token_index(t, e - 1) + 1
    size = max(WINDOW, i1 - i0 + 64)
    if size > MAX_WINDOW:
        return None
    size = min(size, n)
    lo, hi = max(0, i1 - size), min(i0, n - size)
    start = rng.randint(lo, hi) if lo <= hi else max(0, n - size)
    return make(row, start, start + size)

def negative(row, rng, tries=20):
    n = len(token_starts[row.title])
    size = min(WINDOW, n)
    for _ in range(tries):
        start = rng.randint(0, n - size)
        ex = make(row, start, start + size)
        if ex and ex["output"] == NOT_FOUND:
            return ex
    return None

def build(rows, rng, extra_unanswered, extra_answered):
    answered, unanswered = rows[rows.has_answer], rows[~rows.has_answer]
    pos = [positive(r, rng) for r in answered.itertuples()]
    neg_u = [negative(r, rng) for r in unanswered.sample(min(extra_unanswered, len(unanswered)), random_state=SEED).itertuples()]
    neg_a = [negative(r, rng) for r in answered.sample(min(extra_answered, len(answered)), random_state=SEED).itertuples()]
    out = [x for x in pos + neg_u + neg_a if x]
    rng.shuffle(out)
    return out

rng = random.Random(SEED)
train_examples = build(train[~train.title.isin(val_titles)], rng, EXTRA_NEG_FROM_UNANSWERED, EXTRA_NEG_FROM_ANSWERED)
val_examples = build(train[train.title.isin(val_titles)], rng, 150, 150)[:VAL_EXAMPLES]
print(len(train_examples), "train examples,", len(val_examples), "validation examples")
print(json.dumps(train_examples[0], indent=1)[:1500])

In [ ]:
TOKENS_PER_SECOND = 350

def prompt_of(ex):
    return ALPACA.format(ex["instruction"], ex["input"])

def encode(ex):
    p = tokenizer(prompt_of(ex))["input_ids"]
    r = tokenizer(ex["output"], add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    return p + r, [-100] * len(p) + r

train_data = [d for d in map(encode, train_examples) if len(d[0]) <= MAX_TOKENS]
val_data = [d for d in map(encode, val_examples) if len(d[0]) <= MAX_TOKENS]
total_tokens = sum(len(ids) for ids, _ in train_data)
report["data"] = {
    "contracts_train": len(contracts) - VAL_CONTRACTS,
    "contracts_validation": VAL_CONTRACTS,
    "train_examples": len(train_data),
    "validation_examples": len(val_data),
    "train_share_not_found": round(sum(x["output"] == NOT_FOUND for x in train_examples) / len(train_examples), 3),
    "mean_tokens_per_example": round(total_tokens / len(train_data)),
    "total_train_tokens_millions": round(total_tokens / 1e6, 2),
    "estimated_hours": round(total_tokens / TOKENS_PER_SECOND / 3600, 1),
}
print(report["data"])

In [ ]:
import bitsandbytes as bnb
from transformers import get_linear_schedule_with_warmup

ACCUM = 8
LR = 2e-4
EVAL_EVERY = 250
SAVE_EVERY = 250
PRINT_EVERY = 25
TIME_LIMIT_HOURS = 10.5
CHECKPOINT = f"{OUT}/checkpoint"

params = [p for p in model.parameters() if p.requires_grad]
total_steps = len(train_data) // ACCUM
optimizer = bnb.optim.AdamW8bit(params, lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * total_steps), total_steps)
FastLanguageModel.for_training(model)

def batch(ids, labels):
    return torch.tensor([ids], device="cuda"), torch.tensor([labels], device="cuda")

def validation_loss():
    with torch.no_grad():
        losses = [model(input_ids=x, labels=y).loss.item() for x, y in (batch(*d) for d in val_data)]
    return round(sum(losses) / len(losses), 4)

log, val_log = [], [{"step": 0, "val_loss": validation_loss()}]
print("validation loss before training:", val_log[0]["val_loss"])
start, step, running, stopped_early = time.time(), 0, 0.0, False

for i, d in enumerate(train_data[: total_steps * ACCUM]):
    x, y = batch(*d)
    loss = model(input_ids=x, labels=y).loss / ACCUM
    loss.backward()
    running += loss.item()
    if (i + 1) % ACCUM:
        continue
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    step += 1
    hours = (time.time() - start) / 3600
    log.append({"step": step, "train_loss": round(running, 4), "hours": round(hours, 3)})
    running = 0.0
    if step % PRINT_EVERY == 0:
        recent = sum(e["train_loss"] for e in log[-PRINT_EVERY:]) / PRINT_EVERY
        eta = hours / step * (total_steps - step)
        print(f"step {step}/{total_steps}  train_loss {recent:.4f}  elapsed {hours:.2f}h  remaining ~{eta:.2f}h")
    if step % EVAL_EVERY == 0 or step == total_steps:
        val_log.append({"step": step, "val_loss": validation_loss()})
        print("validation loss:", val_log[-1]["val_loss"])
    if step % SAVE_EVERY == 0:
        model.save_pretrained(CHECKPOINT)
        json.dump({"log": log, "val_log": val_log}, open(f"{OUT}/training_log.json", "w"))
    if hours > TIME_LIMIT_HOURS:
        stopped_early = True
        break

report["training"] = {
    "steps_done": step,
    "steps_planned": total_steps,
    "stopped_early": stopped_early,
    "hours": round((time.time() - start) / 3600, 2),
    "train_loss_first_50": round(sum(e["train_loss"] for e in log[:50]) / max(1, len(log[:50])), 4),
    "train_loss_last_50": round(sum(e["train_loss"] for e in log[-50:]) / max(1, len(log[-50:])), 4),
    "validation_loss": val_log,
}
print(report["training"])

In [ ]:
ADAPTER = f"{OUT}/legal-llama-lora"
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
json.dump({"log": log, "val_log": val_log}, open(f"{OUT}/training_log.json", "w"))

FastLanguageModel.for_inference(model)
picks = [x for x in val_examples if x["output"] != NOT_FOUND][:2] + [x for x in val_examples if x["output"] == NOT_FOUND][:1]
report["samples"] = []
for ex in picks:
    inputs = tokenizer(prompt_of(ex), return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        temperature=None,
        top_p=None,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    got = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    report["samples"].append({"question": ex["instruction"].split("\n")[0][:150], "expected": ex["output"][:300], "got": got[:300]})
    print(report["samples"][-1])

In [ ]:
print(json.dumps(report, indent=1, ensure_ascii=False))